#Gold Layer - Customer Dimension

Create a business-ready Customer Dimension from the validated Silver customer data.

**Source:** `end-to-end_pipeline.silver.customers`

**Target:** `end-to-end_pipeline.gold.dim_customer`

**Model Role:** Dimension Table

**Business Key:** `customer_id`

**Approach:** Profile → Inspect → Transform → Validate

**Purpose:**
Provide a consistent customer dimension for analyzing sales by customer characteristics such as location, segment, gender, loyalty status, and registration information.


## Cell 1 - Profile Silver Customer Data

**Description:**
Confirm that the validated Silver customer table is ready to become a Gold dimension. This is **not another data-cleaning step**. It verifies the dimension grain, customer-key uniqueness, row count, and availability of the descriptive attributes required for analytics.

In [0]:
%sql

-- ============================================================
-- CELL 1: PROFILE SILVER CUSTOMERS FOR GOLD MODELING
-- Purpose: Confirm dimension grain, key uniqueness,
--          and availability of business attributes
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS distinct_customer_ids,
    COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_customer_ids,

    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
        AS null_customer_ids,

    COUNT(DISTINCT customer_segment)
        AS customer_segments,

    COUNT(DISTINCT loyalty_status)
        AS loyalty_statuses,

    COUNT(DISTINCT country)
        AS countries,

    MIN(registration_date)
        AS earliest_registration_date,

    MAX(registration_date)
        AS latest_registration_date

FROM `end-to-end_pipeline`.silver.customers;

## Cell 2 - Inspect Business Attributes

**Description:**
Review the main descriptive attributes that will be exposed through the Gold Customer Dimension. This helps confirm that the dimension supports useful analytical slicing without repeating Silver-layer cleaning.

In [0]:
%sql

-- ============================================================
-- CELL 2: INSPECT CUSTOMER BUSINESS ATTRIBUTES
-- Purpose: Review customer segments and loyalty distribution
--          before creating the Gold dimension
-- ============================================================

SELECT
    customer_segment,
    loyalty_status,
    COUNT(*) AS customer_count

FROM `end-to-end_pipeline`.silver.customers

GROUP BY
    customer_segment,
    loyalty_status

ORDER BY
    customer_segment,
    loyalty_status;

## Cell 3 - Transform Silver → Gold Customer Dimension

**Description:**
Create the Gold Customer Dimension at **one row per customer**.

Unlike Silver, this step does not trim, deduplicate, repair NULLs, or standardize values again. The Silver table has already handled those responsibilities.

The Gold table exposes the descriptive customer attributes required for business analytics while preserving `customer_id` as the key that will connect this dimension to `gold.fact_sales`.


In [0]:
%sql

-- ============================================================
-- CELL 3: CREATE GOLD CUSTOMER DIMENSION
-- Grain: One row per customer
-- Business Key: customer_id
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.gold.dim_customer
COMMENT 'Business-ready customer dimension. One row per customer. Source: silver.customers. Used by dashboards, Genie, and fact_sales joins.'
AS

SELECT
    customer_id,
    customer_name,
    gender,
    customer_segment,
    city,
    country,
    registration_date,
    YEAR(registration_date) AS registration_year,
    loyalty_status

FROM `end-to-end_pipeline`.silver.customers;

ALTER TABLE `end-to-end_pipeline`.gold.dim_customer ALTER COLUMN customer_id COMMENT 'Unique customer business key; joins to fact_sales.customer_id';
ALTER TABLE `end-to-end_pipeline`.gold.dim_customer ALTER COLUMN customer_name COMMENT 'Full customer display name';
ALTER TABLE `end-to-end_pipeline`.gold.dim_customer ALTER COLUMN gender COMMENT 'Customer gender (Male / Female / Other)';
ALTER TABLE `end-to-end_pipeline`.gold.dim_customer ALTER COLUMN customer_segment COMMENT 'Customer segment: Consumer, Corporate, or Small Business';
ALTER TABLE `end-to-end_pipeline`.gold.dim_customer ALTER COLUMN city COMMENT 'Customer city of residence';
ALTER TABLE `end-to-end_pipeline`.gold.dim_customer ALTER COLUMN country COMMENT 'Customer country of residence';
ALTER TABLE `end-to-end_pipeline`.gold.dim_customer ALTER COLUMN registration_date COMMENT 'Date the customer registered';
ALTER TABLE `end-to-end_pipeline`.gold.dim_customer ALTER COLUMN registration_year COMMENT 'Derived: year of registration, useful for cohort analysis in dashboards';
ALTER TABLE `end-to-end_pipeline`.gold.dim_customer ALTER COLUMN loyalty_status COMMENT 'Loyalty tier: Bronze, Silver, or Gold';

## Cell 4 - Validate Gold Customer Dimension

**Description:**
Validate the final Customer Dimension and confirm its intended grain: exactly one row per customer. The validation also confirms that every Gold customer originates from the validated Silver customer table.


In [0]:
%sql

-- ============================================================
-- CELL 4: VALIDATE GOLD CUSTOMER DIMENSION
-- Purpose: Confirm dimension grain, key integrity,
--          row consistency, and Silver → Gold completeness
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT customer_id)
            AS distinct_customer_ids,

        COUNT(*) - COUNT(DISTINCT customer_id)
            AS duplicate_customer_ids,

        SUM(
            CASE
                WHEN customer_id IS NULL THEN 1
                ELSE 0
            END
        ) AS null_customer_ids,

        SUM(
            CASE
                WHEN customer_segment IS NULL THEN 1
                ELSE 0
            END
        ) AS null_customer_segments,

        SUM(
            CASE
                WHEN loyalty_status IS NULL THEN 1
                ELSE 0
            END
        ) AS null_loyalty_status,

        SUM(
            CASE
                WHEN country IS NULL THEN 1
                ELSE 0
            END
        ) AS null_countries

    FROM `end-to-end_pipeline`.gold.dim_customer
),

source_check AS (

    SELECT
        COUNT(*) AS silver_rows

    FROM `end-to-end_pipeline`.silver.customers
)

SELECT
    v.*,
    s.silver_rows,

    CASE
        WHEN v.total_rows = s.silver_rows
            AND v.total_rows = v.distinct_customer_ids
            AND v.duplicate_customer_ids = 0
            AND v.null_customer_ids = 0
            AND v.null_customer_segments = 0
            AND v.null_loyalty_status = 0
            AND v.null_countries = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation v
CROSS JOIN source_check s;